# 22DM015 Final Project — Financial PhraseBank
## Person D: Data prep + Part 2 augmentation/generation + Part 4 distillation

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453), `labeled_32` (32).‍
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).‍
- Part 2 'unlabelled' data = train minus the 32 (`unlabeled_pool`).‍
- Evaluate headline numbers on **`test`** only; tune on **`val`**.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

This notebook is the **source of truth for the data**.‍ Running the build cell regenerates the CSVs under `data/` that A/B/C consume.‍ Commit those CSVs so everyone is byte-identical.‍

In [2]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical data for everyone
train, val, test = splits['train'], splits['val'], splits['test']
labeled_32 = splits['labeled_32']
pool = du.unlabeled_pool(splits)     # Part 2 'unlabelled' data
PERSON = 'D'
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

train        1584 {'neutral': np.int64(973), 'positive': np.int64(399), 'negative': np.int64(212)}
val           227 {'neutral': np.int64(140), 'positive': np.int64(57), 'negative': np.int64(30)}
test          453 {'neutral': np.int64(278), 'positive': np.int64(114), 'negative': np.int64(61)}
labeled_32     32 {'positive': np.int64(11), 'negative': np.int64(11), 'neutral': np.int64(10)}
Last run: 2026-06-10 15:40:13


In [3]:
import pandas as pd

#multiple cell outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

Last run: 2026-06-10 15:41:13


## 0.‍ Build the shared splits (run once, then commit `data/`)
Stratified 70/10/20 with SEED=618; balanced 32-shot (11 neg / 10 neu / 11 pos) from train.‍ Logic lives in `data_utils.build_splits()` so it's auditable and reproducible.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Regenerate the canonical CSVs. Safe to re-run: deterministic given SEED.
splits = du.build_splits()
for k, v in splits.items():
    print(f'{k:11s} n={len(v):4d}', dict(v['label_name'].value_counts().reindex(du.LABEL_NAMES)))
# sanity checks
tr, te, va = set(splits['train'].id), set(splits['test'].id), set(splits['val'].id)
assert not (tr & te) and not (tr & va) and not (te & va), 'split leakage!'
assert set(splits['labeled_32'].id) <= tr, '32-shot must come from train'
print('OK: no leakage, 32-shot ⊂ train')

## Part 2b.‍ Dataset Augmentation WITHOUT an LLM (1)

additional notes -- finance dictionary, do currencies carry sentiment? need to add narrative that specifically did not clean data, because the cleaning to apply during modeling. 

Our goal is to augment the 32 financial texts, meaning we want to create more examples out of the ones we have without changing what sentiment each text expresses. To do that well, we have some observations on the characteristics of financial text in relation to sentiments:

1. Some words mean something different in finance than in everyday English. "Interest," for example, means one thing in a finance context and another in general use, so we have to be careful that any change we make keeps the financial meaning.
2. Financial text uses a lot of numbers, and those numbers can themselves carry the sentiment. A movement from one percentage to another can be good news or bad news depending on the direction, so we cannot change numbers carelessly.
3. A lot of business jargon is generic and carries no sentiment at all, like "fourth quarter" or the currency a figure is written in.
4. Some financial jargon does carry sentiment, where a word like "revenue" leans positive and "losses" leans negative.

Based on this, we first thought of making small, targeted changes to the text. We could perturb the numbers while keeping the direction the same, so an increase stays an increase and a decrease stays a decrease. We could also swap the currency for another one, since the currency itself does not change the sentiment. The idea with these edits is to prevent the model from leaning on a specific number or currency as if it were a clue to the sentiment, which is a real danger when we only have 32 examples to learn from.

However, the problem is that these edits only change one small piece at a time, so they barely make the text more diverse. We thought about whether changing more words at once would help, but even then the sentence stays the same in shape and only the vocabulary shifts a little. What we really want is to rephrase the whole sentence, and what helps us do that is back-translation. 

Back-translation works by taking a sentence, translating it into another language, and then translating it back into the original language. This is the kind of thing you can do with Google Translate. Because the translation almost never comes back word for word, what we get is a paraphrase of the original sentence, with the same meaning but different wording and structure. This is not text generated by an LLM. Back-translation uses a neural machine translation model, which is an encoder-decoder network built specifically to turn one language into another. It reworks a sentence we already have rather than inventing new content from a prompt the way a generative model would.

To generate multiple paraphrased sentences of the same text, we can translate through several different languages. We focus on major languages like German, French, and Spanish, and the reason is simply translation quality. We think thse languages have stronger translation models behind them, so the round trip keeps the meaning intact, whereas smaller or very different languages tend to introduce mistakes, and those mistakes usually land on the financial details we most want to protect. So the choice of language is about how reliable the translation is, not about the language having anything to do with finance.

One more thing we considered is masking, where we hide certain words so the translator leaves them alone. This is only worth doing for the parts whose exact value matters and that translation might quietly change, such as numbers, percentages, and currency amounts. It is not needed for generic terms like "fourth quarter," because if those get reworded it does no harm. And for 32 texts, words that should be masked can be manually determined.

And finally of course we should have some kind of measure on how well the diversification is basically how many new words or some kind of similarity measure from the original



In [5]:
df = pd.read_csv("../data/labeled_32.csv")
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          32 non-null     int64
 1   text        32 non-null     str  
 2   label       32 non-null     int64
 3   label_name  32 non-null     str  
dtypes: int64(2), str(2)
memory usage: 4.9 KB


,id,text,label,label_name
0,58,"In the fourth quarter of 2008 , net sales incr...",2,positive
1,65,Operating profit margin increased from 11.2 % ...,2,positive
2,73,The Department Store Division reported an incr...,2,positive
3,85,Advertising and circulation revenues grew by 3...,2,positive
4,125,First quarter underlying operating profit rose...,2,positive


Last run: 2026-06-10 15:41:23


In [7]:
# Print examples of each label
labels = df['label_name'].unique()
for label in labels:
    print(f"\nLabel: {label}")
    examples = df[df['label_name'] == label]['text'].tolist()
    for i, example in enumerate(examples[:5]):  # Print first 3 examples for each label
        print(f"Example {i+1}: {example}")


Label: positive
Example 1: In the fourth quarter of 2008 , net sales increased by 2 % to EUR 1,050.7 mn from EUR 1,027.0 mn in the fourth quarter of 2007 .
Example 2: Operating profit margin increased from 11.2 % to 11.7 % .
Example 3: The Department Store Division reported an increase in sales of 4 per cent .
Example 4: Advertising and circulation revenues grew by 3.4 % and by 0.4 % , respectively .
Example 5: First quarter underlying operating profit rose to 41 mln eur from 33 mln a year earlier .

Label: negative
Example 1: Jan. 6 -- Ford is struggling in the face of slowing truck and SUV sales and a surfeit of up-to-date , gotta-have cars .
Example 2: Result before taxes decreased to nearly EUR 14.5 mn , compared to nearly EUR 20mn in the previous accounting period .
Example 3: Finnish shipping company Finnlines ' pretax loss totalled EUR 6.5 mn in the third quarter of 2009 , compared to a profit of EUR 0.3 mn in the third quarter of 2008 .
Example 4: Operating loss totaled EUR 0.

### Stage 1 — Evidence for the four observations

The cells below quantify each observation on the 32 labelled texts: (1) finance words that also carry non-financial WordNet senses, (2) numbers/percentages and direction words by class, (3) generic jargon spread across classes, (4) sentiment-bearing terms by class. Outputs are raw counts and examples only; with n=32 these are illustrative, not statistical.

In [8]:
# watermark: AGLLM (AI-assisted content disclosure)
# Observation 1: several finance terms also carry non-financial WordNet senses, so naive
# synonym-replacement augmentation can silently change meaning. Show senses + presence in the 32.
import nltk
from nltk.corpus import wordnet as wn
try:
    wn.synsets('test')
except LookupError:
    nltk.download('wordnet'); nltk.download('omw-1.4')

candidates = ['interest', 'share', 'shares', 'bond', 'security', 'margin',
              'return', 'principal', 'capital', 'stock', 'note']
rows = []
for w in candidates:
    in_32 = bool(df['text'].str.contains(rf'\b{w}\b', case=False, regex=True).any())
    syns = []
    for s in wn.synsets(w):
        for lem in s.lemmas():
            name = lem.name().replace('_', ' ')
            if name.lower() != w and name not in syns:
                syns.append(name)
    rows.append({'term': w, 'in_32_texts': in_32,
                 'n_wordnet_senses': len(wn.synsets(w)), 'sample_synonyms': ', '.join(syns[:6])})
pd.DataFrame(rows).sort_values('in_32_texts', ascending=False).reset_index(drop=True)

[Synset('trial.n.02'),
 Synset('test.n.02'),
 Synset('examination.n.02'),
 Synset('test.n.04'),
 Synset('test.n.05'),
 Synset('test.n.06'),
 Synset('test.v.01'),
 Synset('screen.v.01'),
 Synset('quiz.v.01'),
 Synset('test.v.04'),
 Synset('test.v.05'),
 Synset('test.v.06'),
 Synset('test.v.07')]

,term,in_32_texts,n_wordnet_senses,sample_synonyms
0,share,True,10,"portion, part, percentage, parcel, contributio..."
1,shares,True,10,"share, portion, part, percentage, parcel, cont..."
2,margin,True,6,"border, perimeter, security deposit, gross pro..."
3,capital,True,11,"working capital, capital letter, uppercase, up..."
4,stock,True,27,"inventory, gunstock, stock certificate, store,..."
5,interest,False,10,"involvement, sake, interestingness, stake, int..."
6,bond,False,14,"chemical bond, bond certificate, alliance, bai..."
7,security,False,9,"protection, certificate, surety, security depa..."
8,return,False,29,"tax return, income tax return, homecoming, com..."
9,principal,False,7,"school principal, head teacher, head, star, le..."


Last run: 2026-06-10 17:08:00


In [9]:
# watermark: AGLLM (AI-assisted content disclosure)
# Observation 2: numbers/percentages are pervasive; direction words (up/down) track sentiment.
import re
NUM = re.compile(r'\d[\d.,]*\s*%?')
df['n_numbers'] = df['text'].apply(lambda t: len(NUM.findall(t)))
df['has_number'] = df['n_numbers'] > 0
df['dir_up'] = df['text'].str.contains(
    r'\b(?:increase|increased|rose|grew|growth|up|higher|gain|gained)\b', case=False, regex=True)
df['dir_down'] = df['text'].str.contains(
    r'\b(?:decrease|decreased|fell|fall|loss|lower|decline|declined|down|drop|dropped)\b', case=False, regex=True)
df.groupby('label_name').agg(
    texts=('text', 'size'),
    with_numbers=('has_number', 'sum'),
    avg_numbers=('n_numbers', 'mean'),
    up_words=('dir_up', 'sum'),
    down_words=('dir_down', 'sum')).round(2).reindex(du.LABEL_NAMES)

,texts,with_numbers,avg_numbers,up_words,down_words
label_name,,,,,
negative,11,11,2.82,1,10
neutral,10,5,1.40,1,0
positive,11,10,2.64,7,1


Last run: 2026-06-10 17:08:11


In [10]:
# watermark: AGLLM (AI-assisted content disclosure)
# Observation 3: generic business jargon appears across all classes (little sentiment signal).
def class_counts(term):
    m = df['text'].str.contains(rf'\b{term}\b', case=False, regex=True)
    return df.loc[m, 'label_name'].value_counts().reindex(du.LABEL_NAMES).fillna(0).astype(int)

generic_terms = ['quarter', 'year', 'period', 'company', 'eur', 'mn', 'million', 'market']
generic_tbl = pd.DataFrame({t: class_counts(t) for t in generic_terms}).T
generic_tbl['total'] = generic_tbl.sum(axis=1)
generic_tbl.sort_values('total', ascending=False)

label_name,negative,neutral,positive,total
eur,7,1,4,12
mn,7,1,2,10
quarter,3,0,2,5
year,3,0,2,5
period,3,0,2,5
company,1,0,3,4
market,1,1,1,3
million,1,0,0,1


Last run: 2026-06-10 17:08:16


In [11]:
# watermark: AGLLM (AI-assisted content disclosure)
# Observation 4: sentiment-bearing finance terms concentrate in particular classes.
# (Reuses class_counts defined in the Observation 3 cell — run that cell first.)
sentiment_terms = ['profit', 'growth', 'increased', 'rose', 'gain',
                   'loss', 'decline', 'fell', 'decreased', 'lower']
sentiment_tbl = pd.DataFrame({t: class_counts(t) for t in sentiment_terms}).T
sentiment_tbl['total'] = sentiment_tbl.sum(axis=1)
sentiment_tbl.sort_values('total', ascending=False)

label_name,negative,neutral,positive,total
profit,7,0,3,10
increased,0,0,3,3
loss,2,0,1,3
decreased,2,0,0,2
rose,0,0,1,1
fell,1,0,0,1
growth,0,0,0,0
gain,0,0,0,0
decline,0,0,0,0
lower,0,0,0,0


Last run: 2026-06-10 17:08:20


### Reading of the evidence (Observations 1–4)

_TODO (student-authored): interpret the tables above in your own words — what each one does or does not support, and what it implies for how we may safely augment._

## Stage 2 — Generate paraphrases by back-translation (no masking)

Each of the 32 texts is round-tripped through seven languages, one per family — Spanish (Romance), German (Germanic), Russian (Slavic), Finnish (Uralic), Tagalog (Austronesian), Arabic (Semitic), Chinese (Sino-Tibetan) — using Helsinki-NLP `opus-mt` neural MT models (encoder–decoder translators, not generative LLMs). We do not mask; one uniform filter keeps a paraphrase only if it preserves the source digits, differs from the original, is not a duplicate, and does not leak into val/test. Decoding is beam search (deterministic, no sampling) and the model revisions are pinned, so a re-run reproduces given the same environment; the resulting `data/augmented_32.csv` is committed as the shared artifact.

### Back-translation parameters (used to build `data/augmented_32.csv`)

| parameter | value | what it controls |
|---|---|---|
| pivot languages | `es, de, ru, fi, tl, ar, zh` (7) | round-trip languages, one per family |
| decoding strategy | beam search | deterministic decoding (no randomness) |
| `num_beams` | 4 | beam width |
| `do_sample` | `False` | random sampling disabled |
| `num_return_sequences` | 1 | paraphrases generated per text per pivot |
| `max_length` | 256 | max tokens per generation / truncation length |
| `batch_size` | 16 | sentences per forward pass |
| model revisions | pinned (14 commit hashes) | exact opus-mt versions (see next cell) |
| keep-filter | numbers preserved · differs from source · deduplicated · no val/test leakage | candidate acceptance rule |

_TODO (student-authored): justify these choices for this task — e.g. beam vs sampling (determinism plus the diversity/fidelity trade-off observed in our experiments), the seven-pivot one-per-family selection, and the number-preserving filter._

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Back-translation via Helsinki-NLP opus-mt (neural MT, NOT an LLM). Seven pivots, one per
# family: es (Romance), de (Germanic), ru (Slavic), fi (Uralic), tl (Austronesian),
# ar (Semitic), zh (Sino-Tibetan). Model revisions are PINNED so the run is reproducible.
import torch, transformers
from transformers import MarianMTModel, MarianTokenizer

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
PIVOTS = ['es', 'de', 'ru', 'fi', 'tl', 'ar', 'zh']
# Exact opus-mt commit hashes used to build the committed data/augmented_32.csv.
REVISIONS = {
    'Helsinki-NLP/opus-mt-en-es': '5bc4493d463cf000c1f0b50f8d56886a392ed4ab',
    'Helsinki-NLP/opus-mt-es-en': 'c96e2c5399ebfae4fc43d9669556b9afa74bb69d',
    'Helsinki-NLP/opus-mt-en-de': '6183067f769a302e3861815543b9f312c71b0ca4',
    'Helsinki-NLP/opus-mt-de-en': '1a922f3b32a8e809e17a47d4b32142d8105924e5',
    'Helsinki-NLP/opus-mt-en-ru': 'bb09c99d180016eac6819df3dae68edb1690fdee',
    'Helsinki-NLP/opus-mt-ru-en': 'fbd6dc73284f95536648512cc21d57f19191961a',
    'Helsinki-NLP/opus-mt-en-fi': '2f68f12519005b50f9e1afe6fd57b13add55e156',
    'Helsinki-NLP/opus-mt-fi-en': 'c4d8fff0f8f00b637f0578666f35cdacef354778',
    'Helsinki-NLP/opus-mt-en-tl': 'e46e1761492cb6a6fb9515a72bb55ca654815ca5',
    'Helsinki-NLP/opus-mt-tl-en': '8d2dee91f8b3fa48dda8ec06ee2af8f24970dcbc',
    'Helsinki-NLP/opus-mt-en-ar': '03087980e8ce753d64b3248ed0a912444545b840',
    'Helsinki-NLP/opus-mt-ar-en': 'c5b2a50db78d6be98ae207223f8d4f63bc7a0ff1',
    'Helsinki-NLP/opus-mt-en-zh': '408d9bc410a388e1d9aef112a2daba955b945255',
    'Helsinki-NLP/opus-mt-zh-en': 'cf109095479db38d6df799875e34039d4938aaa6',
}
_MODELS = {}

@torch.no_grad()
def translate(texts, model_name, batch_size=16, num_beams=4):
    if model_name not in _MODELS:
        rev = REVISIONS.get(model_name)
        tok = MarianTokenizer.from_pretrained(model_name, revision=rev)
        model = MarianMTModel.from_pretrained(model_name, revision=rev).to(DEVICE).eval()
        _MODELS[model_name] = (tok, model)
    tok, model = _MODELS[model_name]
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors='pt',
                  padding=True, truncation=True, max_length=256).to(DEVICE)
        gen = model.generate(**enc, num_beams=num_beams, max_length=256)  # beam = deterministic
        out.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return out

def back_translate(texts, pivot):
    return translate(translate(texts, f'Helsinki-NLP/opus-mt-en-{pivot}'),
                     f'Helsinki-NLP/opus-mt-{pivot}-en')  # en -> pivot -> en

# Provenance for reproducibility: record what produced the committed CSV.
print(f'device={DEVICE} | torch={torch.__version__} | transformers={transformers.__version__}')
print('pivots:', PIVOTS, '| 14 model revisions pinned')

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Uniform filter (one rule for every language): keep a paraphrase only if it preserves the
# source digits, differs from the source, is not a duplicate, and does not leak into val/test.
# `clean` is a language-neutral normalization that decodes HTML entities (e.g. &apos; -> ').
import re, html
_DIGITS = re.compile(r'\d+')
def digit_multiset(t): return sorted(_DIGITS.findall(str(t)))   # ignores separators/decimal style
def norm(t): return ' '.join(str(t).lower().split())
def numbers_preserved(src, cand): return digit_multiset(src) == digit_multiset(cand)
def clean(t): return re.sub(r'\s{2,}', ' ', html.unescape(str(t))).strip()

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Run back-translation over the 32 through every pivot, apply the uniform filter, write the CSV.
src_texts = labeled_32['text'].tolist()
records = []
for pivot in PIVOTS:
    bt = back_translate(src_texts, pivot)
    for (_, row), cand in zip(labeled_32.iterrows(), bt):
        records.append({'source_id': row['id'], 'method': f'bt-{pivot}',
                        'label': row['label'], 'label_name': row['label_name'],
                        'src': row['text'], 'text': clean(cand)})
cand_df = pd.DataFrame(records)
cand_df['numbers_ok'] = [numbers_preserved(s, c) for s, c in zip(cand_df['src'], cand_df['text'])]
cand_df['is_new'] = cand_df['text'].map(norm) != cand_df['src'].map(norm)

orig_norm = set(map(norm, src_texts))
kept = (cand_df[cand_df['numbers_ok'] & cand_df['is_new']]
        .loc[lambda d: ~d['text'].map(norm).isin(orig_norm)]
        .drop_duplicates(subset='text'))

# guard against leakage into the held-out splits
forbidden = set(pd.concat([val['text'], test['text']]).map(norm))
assert kept['text'].map(norm).isin(forbidden).sum() == 0, 'augmentation leaked into val/test'

orig = labeled_32[['id', 'text', 'label', 'label_name']].assign(method='original', source_id=labeled_32['id'])
new = kept[['text', 'label', 'label_name', 'method', 'source_id']].copy()
new.insert(0, 'id', range(100000, 100000 + len(new)))
augmented = pd.concat([orig, new], ignore_index=True)[['id', 'text', 'label', 'label_name', 'method', 'source_id']]
augmented.to_csv('../data/augmented_32.csv', index=False)

print('kept per pivot:'); print(kept['method'].value_counts())
print(f"\nnumbers_ok: {int(cand_df['numbers_ok'].sum())}/{len(cand_df)}  |  total kept: {len(kept)}")
print(f"augmented_32.csv: {len(augmented)} rows = {len(orig)} originals + {len(new)} paraphrases")
print(augmented['label_name'].value_counts().reindex(du.LABEL_NAMES))

## Stage 3 — View the augmented dataset

The cells below describe the back-translated set saved in `data/augmented_32.csv`: original-vs-paraphrase samples, size/class/method counts, integrity checks, a token-overlap (Jaccard) diversity measure, and how often currency/unit tokens changed (a side effect of not masking). Outputs are descriptive only.

In [19]:
# watermark: AGLLM (AI-assisted content disclosure)
# Original vs its paraphrases — one source example per class.
aug = pd.read_csv('../data/augmented_32.csv')
src_text = labeled_32.set_index('id')['text']
example_ids = aug[aug.method == 'original'].groupby('label_name', group_keys=False).head(5)['source_id']
for sid in example_ids:
    grp = aug[aug.source_id == sid]
    print(f"\n=== source {sid} [{grp.iloc[0]['label_name']}] ===")
    print("ORIG:", src_text.get(sid))
    for _, r in grp[grp.method != 'original'].iterrows():
        print(f"[{r['method']}] {r['text']}")


=== source 58 [positive] ===
ORIG: In the fourth quarter of 2008 , net sales increased by 2 % to EUR 1,050.7 mn from EUR 1,027.0 mn in the fourth quarter of 2007 .
[bt-es] In the fourth quarter of 2008 net sales increased by 2 % to € 1,050.7 million compared to € 1,027.0 million in the fourth quarter of 2007 .
[bt-de] In the fourth quarter of 2008 net sales increased by 2% from EUR 1,027.0 million in the fourth quarter of 2007 to EUR 1,050.7 million .
[bt-tl] At the fourth quarter of 2008 , the net sales rose to 2 %s in EUR 1,050.7 mn from EUR 1,027.0 mn in the fourth quarter quarter of 2007 .
[bt-ar] In the fourth quarter of 2008, net sales increased by 2 per cent to Euro1,050.7 million from Euro1,027.0 million in the fourth quarter of 2007.

=== source 65 [positive] ===
ORIG: Operating profit margin increased from 11.2 % to 11.7 % .
[bt-es] The operating profit margin increased from 11,2 % to 11,7 % .
[bt-de] The operating profit margin increased from 11.2% to 11.7% .
[bt-ru] The op

In [17]:
# watermark: AGLLM (AI-assisted content disclosure)
# Counts and integrity checks on the saved augmented set.
import re
norm = lambda t: ' '.join(str(t).lower().split())
_D = re.compile(r'\d+'); digits = lambda t: sorted(_D.findall(str(t)))
para = aug[aug.method != 'original'].copy()

print('rows per method:'); print(aug['method'].value_counts())
print('\nrows per class:'); print(aug['label_name'].value_counts().reindex(du.LABEL_NAMES))
print(f"\nsize: {len(aug)} rows = {(aug.method == 'original').sum()} original + {len(para)} paraphrases "
      f"({len(aug) / 32:.1f}x the 32)")

para['numbers_ok'] = [digits(src_text[s]) == digits(t) for s, t in zip(para['source_id'], para['text'])]
forbidden = set(pd.concat([val['text'], test['text']]).map(norm))
checks = {
    'duplicate texts': int(aug['text'].duplicated().sum()),
    'paraphrases with changed numbers': int((~para['numbers_ok']).sum()),
    'label / label_name mismatches': int((aug.groupby('label')['label_name'].nunique() != 1).sum()),
    'rows leaked into val/test': int(aug['text'].map(norm).isin(forbidden).sum()),
}
print('\nintegrity checks (all should be 0):')
for k, v in checks.items():
    print(f'  {k}: {v}')

rows per method:
method
original    32
bt-es       32
bt-de       31
bt-tl       30
bt-ru       28
bt-fi       28
bt-ar       27
bt-zh       17
Name: count, dtype: int64

rows per class:
label_name
negative    78
neutral     70
positive    77
Name: count, dtype: int64

size: 225 rows = 32 original + 193 paraphrases (7.0x the 32)

integrity checks (all should be 0):
  duplicate texts: 0
  paraphrases with changed numbers: 0
  label / label_name mismatches: 0
  rows leaked into val/test: 0
Last run: 2026-06-10 18:34:28


In [18]:
# watermark: AGLLM (AI-assisted content disclosure)
# Diversity: token-overlap (Jaccard) of each paraphrase with its source (lower = more rewording).
def _toks(t): return set(re.findall(r'[a-z]+', str(t).lower()))
def jaccard(a, b):
    A, B = _toks(a), _toks(b)
    return len(A & B) / len(A | B) if (A | B) else 1.0
para['jaccard_to_src'] = [jaccard(src_text[s], t) for s, t in zip(para['source_id'], para['text'])]
print('mean token Jaccard to source by method:')
print(para.groupby('method')['jaccard_to_src'].mean().round(3))
print('overall mean:', round(para['jaccard_to_src'].mean(), 3))

# Currency/unit drift: we did not mask, so quantify it. Digits are still guarded above.
CUR = re.compile(r'\b(?:eur|usd|gbp|isk|sek|chf|mn|bn|million|billion)\b|[€$£]', re.I)
cur_tokens = lambda t: sorted(c.lower() for c in CUR.findall(str(t)))
para['currency_changed'] = [cur_tokens(src_text[s]) != cur_tokens(t) for s, t in zip(para['source_id'], para['text'])]
print(f"\nparaphrases where currency/unit tokens changed: {int(para['currency_changed'].sum())} / {len(para)}")

mean token Jaccard to source by method:
method
bt-ar    0.512
bt-de    0.684
bt-es    0.666
bt-fi    0.526
bt-ru    0.497
bt-tl    0.539
bt-zh    0.521
Name: jaccard_to_src, dtype: float64
overall mean: 0.57

paraphrases where currency/unit tokens changed: 68 / 193
Last run: 2026-06-10 18:34:33


### Reading of the results — TODO (student-authored)

_Write your assessment here: the diversity-vs-fidelity trade-off, the guard's effect (including any over-rejections like "400,000,000" vs "400 million"), currency/unit drift from not masking, and whether this set is suitable to train Person B's model on._

## Part 2d.‍ Data Generation WITH an LLM (1)
Prompt an LLM to generate new labelled financial sentences (balanced across classes).‍ Save `data/llm_generated.csv` (same schema).‍ Declare the model + watermark.‍ Person B trains on 32 + generated and reports the metric delta.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# TODO: LLM generation. Validate generated labels, dedupe vs train/test to avoid leakage.
# gen.to_csv('../data/llm_generated.csv', index=False)
# Guard against leakage:
# assert gen['text'].isin(set(test['text'])).sum() == 0

## Part 4.‍ Model Distillation / Quantization (3) — start thinking
Take the best model (likely Person B's fine-tuned BERT) and produce a lighter student.‍
- **4a (1.5):** distillation (e.g.‍ DistilBERT student via logit/feature distillation) and/or quantization (dynamic int8 with `torch.quantization`, or bitsandbytes).‍ Document tools.‍
- **4b (0.5):** compare student vs teacher on `test` — macro-F1 *and* inference speed / size.‍
- **4c (1):** analyse where the student degrades; propose improvements.‍
Reuse `eu.evaluate` + `eu.log_result('distilbert-student','distilled',...)` so it lands in the same table.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Placeholder for Part 4 — depends on Person B's trained teacher checkpoint.
# Plan: load teacher, distill to DistilBERT student, then time inference and compare size.
print('Part 4 scaffold — fill in once a teacher checkpoint exists')